# 02 — SEB sin restricciones

Aplicamos los algoritmos de **Seidel** y **Welzl** sobre los accidentes reales
de Manhattan para obtener la Smallest Enclosing Ball libre (sin la restriccion
de Central Park ni los rios).

Este es el **escenario base** del Experimento 1 del proyecto.

In [1]:
import sys
sys.path.insert(0, '..')
import time
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Rectangle
from pyproj import Transformer

from src.preprocessing import cargar_procesado, obtener_puntos
from src.seb_seidel import seb_seidel
from src.seb_welzl import seb_welzl
from src.validar_seb import comparar_bolas, verificar_solucion
from src.data_loader import obtener_zonas_osm

ModuleNotFoundError: No module named 'osmnx'

## 1. Cargar datos y zonas

In [ ]:
df = cargar_procesado()
puntos = obtener_puntos(df)
print(f'Puntos UTM: {puntos.shape}')

# Zonas geograficas para visualizacion
zonas_wgs = obtener_zonas_osm()
print('Zonas:', list(zonas_wgs.keys()))

## 2. Ejecutar ambos algoritmos y comparar

In [ ]:
t0 = time.perf_counter()
centro_seidel, radio_seidel = seb_seidel(puntos, semilla=42)
t_seidel = time.perf_counter() - t0

t0 = time.perf_counter()
centro_welzl, radio_welzl = seb_welzl(puntos, semilla=42)
t_welzl = time.perf_counter() - t0

print(f'Seidel: r = {radio_seidel:.2f} m  centro = ({centro_seidel[0]:.1f}, {centro_seidel[1]:.1f})  [{t_seidel*1000:.1f} ms]')
print(f'Welzl:  r = {radio_welzl:.2f} m  centro = ({centro_welzl[0]:.1f}, {centro_welzl[1]:.1f})  [{t_welzl*1000:.1f} ms]')
print(f'Iguales? {comparar_bolas((centro_seidel, radio_seidel), (centro_welzl, radio_welzl))}')
print(f'Speedup Seidel: {t_welzl/t_seidel:.1f}x')

## 3. Verificar solucion

In [ ]:
ver = verificar_solucion(puntos, (centro_seidel, radio_seidel))
for k, v in ver.items():
    print(f'  {k:25s}: {v}')

## 4. Convertir centro a lat/lon (para interpretar)

In [ ]:
transformer_inv = Transformer.from_crs('EPSG:32618', 'EPSG:4326', always_xy=True)
lon_c, lat_c = transformer_inv.transform(centro_seidel[0], centro_seidel[1])
print(f'Centro SEB en lat/lon: ({lat_c:.5f}, {lon_c:.5f})')
print(f'Radio: {radio_seidel/1000:.2f} km')
print(f'Diametro: {2*radio_seidel/1000:.2f} km')

# El centro deberia caer en algun punto de Manhattan (Midtown / Central Park area)

## 5. Visualizacion en espacio UTM

In [ ]:
# Reproyectar las zonas geograficas a UTM para superponerlas
transformer_dir = Transformer.from_crs('EPSG:4326', 'EPSG:32618', always_xy=True)

def bbox_a_utm(geom):
    lon_min, lat_min, lon_max, lat_max = geom.bounds
    x1, y1 = transformer_dir.transform(lon_min, lat_min)
    x2, y2 = transformer_dir.transform(lon_max, lat_max)
    return (x1, y1, x2-x1, y2-y1)  # x, y, width, height

rect_manh = bbox_a_utm(zonas_wgs['region_factible'])
rect_cp   = bbox_a_utm(zonas_wgs['central_park'])

fig, ax = plt.subplots(figsize=(8, 12))

# Region factible (bbox de Manhattan)
ax.add_patch(Rectangle(rect_manh[:2], rect_manh[2], rect_manh[3],
                       fill=False, edgecolor='blue', linewidth=1.5,
                       linestyle='--', label='Region factible R (bbox Manhattan)'))

# Central Park
ax.add_patch(Rectangle(rect_cp[:2], rect_cp[2], rect_cp[3],
                       facecolor='green', alpha=0.3, edgecolor='darkgreen',
                       linewidth=1.5, label='Central Park (zona prohibida)'))

# Puntos de accidentes
ax.scatter(puntos[:,0], puntos[:,1], s=1, alpha=0.3,
           color='steelblue', label=f'Accidentes (n={len(puntos):,})')

# SEB (sin restricciones)
circulo = Circle(centro_seidel, radio_seidel, fill=False,
                 edgecolor='red', linewidth=2, label=f'SEB libre (r={radio_seidel/1000:.2f} km)')
ax.add_patch(circulo)

# Centro
ax.plot(centro_seidel[0], centro_seidel[1], 'r*', markersize=18,
        markeredgecolor='black', label='Centro SEB libre')

ax.set_xlabel('x UTM (m)')
ax.set_ylabel('y UTM (m)')
ax.set_title('Smallest Enclosing Ball sin restricciones\n(Escenario base — Experimento 1)')
ax.set_aspect('equal')
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../results/figures/02_seb_sin_restricciones.png', dpi=150, bbox_inches='tight')
plt.show()

# Importante: ver si el centro libre cae DENTRO de Central Park
# (eso justifica el experimento 1 con restricciones)

## 6. Test de complejidad O(n) preliminar

Medir tiempos en escala logaritmica para verificar que Seidel se comporta como O(n).

In [ ]:
import pandas as pd
from src.preprocessing import muestra_estratificada

tamanos = [100, 500, 1_000, 2_500, 5_000, 7_947]
n_repeticiones = 5

filas = []
for n in tamanos:
    pts_n = muestra_estratificada(df, n)
    tiempos = []
    for _ in range(n_repeticiones):
        t0 = time.perf_counter()
        seb_seidel(pts_n)
        tiempos.append(time.perf_counter() - t0)
    filas.append({
        'n': n,
        'tiempo_promedio_ms': np.mean(tiempos) * 1000,
        'tiempo_min_ms': np.min(tiempos) * 1000,
        'tiempo_max_ms': np.max(tiempos) * 1000,
    })

df_tiempos = pd.DataFrame(filas)
df_tiempos

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(df_tiempos['n'], df_tiempos['tiempo_promedio_ms'],
            yerr=[df_tiempos['tiempo_promedio_ms']-df_tiempos['tiempo_min_ms'],
                  df_tiempos['tiempo_max_ms']-df_tiempos['tiempo_promedio_ms']],
            marker='o', capsize=4, label='Seidel (medido)')

# Linea de referencia O(n)
ratio = df_tiempos['tiempo_promedio_ms'].iloc[-1] / df_tiempos['n'].iloc[-1]
ax.plot(df_tiempos['n'], ratio * df_tiempos['n'],
        '--', alpha=0.5, label='Referencia O(n)')

ax.set_xlabel('n (numero de puntos)')
ax.set_ylabel('Tiempo (ms)')
ax.set_title('Complejidad temporal de Seidel\n(promedio de 5 ejecuciones por tamano)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../results/figures/02_complejidad_seidel.png', dpi=150)
plt.show()

## 7. Conclusiones de la Fase 2

- Seidel y Welzl producen exactamente el mismo radio y centro.
- Seidel es ~4-5x mas rapido por evitar el overhead recursivo.
- El centro libre cae cerca del centroide geografico de los accidentes.
- **Si el centro libre cae dentro de Central Park, el Experimento 1 sera visualmente impactante**: el algoritmo restringido tendra que desplazarlo al borde sur del parque.